# Phase 3.1 — EDA & Cohort Analysis

**Business Question:** 

- Where is the "leaky bucket"? 
- Which customers are most at risk, and
- what does the data tell us about the scale of the early-churn problem?

**Analyst:** SaaSGuard Platform · Phase 3 EDA
**Reference Date:** 2026-03-14
**Data Source:** `data/saasguard.duckdb` · 5,000 customers · 3.5M usage events

---

> **Narrative-first principle:** 
> 
> Every section opens with a business question and closes
> with a plain-English business answer + exec deck bullet + ROI connection.

## 0 — Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from pathlib import Path

# ── Plotting style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "font.family": "sans-serif",
})
sns.set_palette("husl")

# ── Database connection ────────────────────────────────────────────────────
NOTEBOOKS_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOKS_DIR.parent
DB_PATH = PROJECT_ROOT / "data" / "saasguard.duckdb"
REFERENCE_DATE = "2026-03-14"

conn = duckdb.connect(str(DB_PATH), read_only=True)
print(f"Connected: {DB_PATH.name}  ({DB_PATH.stat().st_size / 1e6:.1f} MB)")
schemas = conn.execute("SHOW SCHEMAS").df()["schema_name"].tolist()
print(f"Schemas: {schemas}")

## 1.1 — Data Load & Sanity Check

> **Business Question:** Does the synthetic data match the designed churn rates?
> Are the key correlations intact?

Reproducing the Phase 2 validation summary to confirm the data we'll analyse
is the same data the warehouse builder ingested.

In [ ]:
# ── Churn rate by plan tier ────────────────────────────────────────────────
tier_summary = conn.execute("""
    SELECT
        plan_tier,
        COUNT(*)                                               AS n_customers,
        COUNT(churn_date)                                      AS n_churned,
        ROUND(COUNT(churn_date) * 100.0 / COUNT(*), 1)        AS churn_rate_pct,
        ROUND(AVG(mrr), 0)                                     AS avg_mrr_usd
    FROM raw.customers
    GROUP BY plan_tier
    ORDER BY avg_mrr_usd DESC
""").df()

print("=== Churn Rate by Plan Tier ===")
print(tier_summary.to_string(index=False))

# ── Event type distribution ────────────────────────────────────────────────
event_summary = conn.execute("""
    SELECT
        event_type,
        COUNT(*)                                              AS n_events,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)   AS pct
    FROM raw.usage_events
    GROUP BY event_type
    ORDER BY n_events DESC
""").df()

print("\n=== Event Type Distribution ===")
print(event_summary.to_string(index=False))

total_events = event_summary["n_events"].sum()
print(f"\nTotal usage events: {total_events:,}")

## 1.2 — The Leaky Bucket: Monthly Cohort Retention

> **Business Question:** How much revenue is being lost in the first 3 months,
> and which signup cohorts are most affected?
>
> **Why this matters:** A leaky bucket means marketing spend fills the funnel while
> the bottom drains. Month-3 retention loss directly sizes the CS opportunity.

**Method:** Monthly cohort table — rows = signup cohort, columns = months since
signup, values = % of cohort still active. A 70% Month-3 retention means 30% of
that cohort churned within 90 days.

In [ ]:
# ── Load all customers with duration ──────────────────────────────────────
customers_df = conn.execute(f"""
    SELECT
        customer_id,
        plan_tier,
        industry,
        signup_date::DATE                                        AS signup_date,
        churn_date::DATE                                         AS churn_date,
        mrr,
        CASE WHEN churn_date IS NOT NULL THEN 1 ELSE 0 END       AS is_churned,
        CASE
            WHEN churn_date IS NOT NULL
                THEN DATEDIFF('day', signup_date::DATE, churn_date::DATE)
            ELSE
                DATEDIFF('day', signup_date::DATE, DATE '{REFERENCE_DATE}')
        END                                                      AS duration_days
    FROM raw.customers
""").df()

customers_df["signup_month"] = (
    pd.to_datetime(customers_df["signup_date"]).dt.to_period("M")
)
customers_df["duration_months"] = (
    (customers_df["duration_days"] / 30.44).apply(np.floor).astype(int)
)

print(f"Total customers:  {len(customers_df):,}")
print(f"Overall churn:    {customers_df['is_churned'].mean():.1%}")
print(f"Signup range:     {customers_df['signup_date'].min()} → {customers_df['signup_date'].max()}")
print()
print(customers_df[["plan_tier","is_churned","duration_days","mrr"]].describe().round(1))

In [ ]:
# ── Build cohort retention table ──────────────────────────────────────────
cohort_months = sorted(customers_df["signup_month"].unique())
month_windows = list(range(0, 13))   # M+00 to M+12

cohort_rows = []
for cohort in cohort_months:
    cohort_cust = customers_df[customers_df["signup_month"] == cohort]
    size = len(cohort_cust)
    row = {"cohort": str(cohort), "size": size}
    for m in month_windows:
        # Retained at month m = survived at least m months (or still active)
        retained = (
            (cohort_cust["is_churned"] == 0) | (cohort_cust["duration_months"] >= m)
        ).sum()
        row[f"M+{m:02d}"] = round(retained / size * 100, 1)
    cohort_rows.append(row)

cohort_df = pd.DataFrame(cohort_rows)
_ = cohort_df.pop("size")
cohort_df = cohort_df.set_index("cohort")
retention_matrix = cohort_df.values.astype(float)

# ── Heatmap ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 8))
cmap = sns.color_palette("RdYlGn", as_cmap=True)
im = ax.imshow(retention_matrix, aspect="auto", cmap=cmap, vmin=0, vmax=100)

ax.set_xticks(range(len(cohort_df.columns)))
ax.set_xticklabels(cohort_df.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(cohort_df.index)))
ax.set_yticklabels(cohort_df.index, fontsize=9)

for i in range(len(cohort_df.index)):
    for j in range(len(cohort_df.columns)):
        val = retention_matrix[i, j]
        color = "black" if val > 50 else "white"
        ax.text(j, i, f"{val:.0f}%", ha="center", va="center", fontsize=7, color=color)

plt.colorbar(im, ax=ax, label="% Retained")
ax.set_title(
    "Monthly Cohort Retention Heatmap\n"
    "Rows = Signup Cohort · Columns = Months Since Signup · Values = % Still Active",
    fontsize=13, fontweight="bold", pad=15,
)
ax.set_xlabel("Months Since Signup", fontsize=11)
ax.set_ylabel("Signup Cohort", fontsize=11)
plt.tight_layout()
plt.show()

# ── Business insight ───────────────────────────────────────────────────────
m3_col = "M+03"
avg_m3 = cohort_df[m3_col].mean()
worst_cohort = cohort_df[m3_col].idxmin()
worst_val = cohort_df.loc[worst_cohort, m3_col]

print(f"\n📊 Business Insight:")
print(f"   Average Month-3 retention:  {avg_m3:.1f}%  (dropout: {100-avg_m3:.1f}%)")
print(f"   Worst cohort at Month-3:    {worst_cohort} → {worst_val:.1f}% retained")
print(f"\n🎯 Exec Deck Bullet:")
print(f"   '{100-avg_m3:.0f}% of a typical signup cohort had churned by month 3 —")
print(f"    the classic SaaS leaky bucket, quantified on our own data.'")

## 1.3 — Plan Tier Churn Rate Distribution

> **Business Question:** Is the enterprise tier truly protected, or does a specific
> industry pull down the average?
>
> **Why this matters:** Tier-differentiated CS investment requires meaningfully different
> churn rates per tier. If industry variation overwhelms tier variation, industry is the
> right segmentation axis for CS outreach budget allocation.

In [ ]:
# ── Aggregate: tier × industry ─────────────────────────────────────────────
tier_industry = customers_df.groupby(["plan_tier", "industry"]).agg(
    n_customers=("customer_id", "count"),
    n_churned=("is_churned", "sum"),
).reset_index()
tier_industry["churn_rate"] = tier_industry["n_churned"] / tier_industry["n_customers"]

tier_agg = customers_df.groupby("plan_tier").agg(
    n=("customer_id", "count"),
    churned=("is_churned", "sum"),
).reset_index()
tier_agg["churn_rate"] = tier_agg["churned"] / tier_agg["n"]

TIER_ORDER = ["starter", "growth", "enterprise"]
tier_agg["plan_tier"] = pd.Categorical(
    tier_agg["plan_tier"], categories=TIER_ORDER, ordered=True
)
tier_agg = tier_agg.sort_values("plan_tier")

# ── Plot ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

colors = {"starter": "#F44336", "growth": "#FF9800", "enterprise": "#4CAF50"}
bars = axes[0].bar(
    tier_agg["plan_tier"],
    tier_agg["churn_rate"] * 100,
    color=[colors[t] for t in tier_agg["plan_tier"]],
    edgecolor="white", linewidth=1.5,
)
for bar, (_, row) in zip(bars, tier_agg.iterrows()):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.5,
        f"{row['churn_rate']:.1%}\n(n={row['n']:,})",
        ha="center", va="bottom", fontsize=11, fontweight="bold",
    )
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].set_title("Overall Churn Rate by Plan Tier", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Plan Tier")
axes[0].set_ylabel("Churn Rate")
axes[0].set_ylim(0, tier_agg["churn_rate"].max() * 135)

pivot = tier_industry.pivot_table(
    index="industry", columns="plan_tier", values="churn_rate", aggfunc="mean",
)[TIER_ORDER]
pivot.plot(kind="bar", ax=axes[1], color=[colors[t] for t in TIER_ORDER], width=0.7)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_title("Churn Rate: Industry × Plan Tier", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Industry")
axes[1].set_ylabel("Churn Rate")
axes[1].legend(title="Plan Tier", loc="upper right")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

starter_churn = tier_agg[tier_agg["plan_tier"] == "starter"]["churn_rate"].values[0]
enterprise_churn = tier_agg[tier_agg["plan_tier"] == "enterprise"]["churn_rate"].values[0]
print(f"\n📊 Starter churn: {starter_churn:.1%}  |  Enterprise churn: {enterprise_churn:.1%}")
print(f"   Tier risk multiplier: {starter_churn/enterprise_churn:.1f}× — starter is highest priority")

## 1.4 — Feature Distribution: Churned vs. Active

> **Business Question:** Can we see the churn signal before modelling?
> Do churned customers look visibly different on key features?
>
> **Why this matters:** If violin plots show no separation, the model has nothing to learn.
> Visual inspection is the fastest sanity check on data quality.

**Note:** This query aggregates 3.5M usage events — allow ~10–20 seconds.

In [ ]:
# ── Load feature data for all customers (churned + active) ─────────────────
feature_df = conn.execute(f"""
    WITH customer_ref AS (
        SELECT
            customer_id,
            plan_tier,
            industry,
            signup_date::DATE                                         AS signup_date,
            CASE WHEN churn_date IS NOT NULL THEN 1 ELSE 0 END        AS is_churned,
            COALESCE(churn_date::DATE, DATE '{REFERENCE_DATE}')        AS reference_date
        FROM raw.customers
    ),
    event_agg AS (
        SELECT
            e.customer_id,
            COUNT(*) FILTER (
                WHERE e.timestamp::DATE >= cr.reference_date - INTERVAL '30 days'
            )                                                         AS events_last_30d,
            AVG(e.feature_adoption_score)                             AS avg_adoption_score,
            COUNT(*) FILTER (
                WHERE e.event_type IN (
                    'evidence_upload', 'monitoring_run', 'report_view'
                )
            )                                                         AS retention_signal_count,
            COUNT(*) FILTER (
                WHERE e.event_type = 'integration_connect'
                  AND e.timestamp::DATE <= cr.signup_date + INTERVAL '30 days'
            )                                                         AS integration_connects_first_30d
        FROM raw.usage_events e
        JOIN customer_ref cr ON e.customer_id = cr.customer_id
        GROUP BY e.customer_id
    ),
    ticket_agg AS (
        SELECT
            customer_id,
            COUNT(*) FILTER (WHERE priority IN ('high', 'critical')) AS high_priority_tickets
        FROM raw.support_tickets
        GROUP BY customer_id
    )
    SELECT
        cr.customer_id, cr.plan_tier, cr.industry, cr.is_churned,
        COALESCE(ea.events_last_30d,          0)    AS events_last_30d,
        COALESCE(ea.avg_adoption_score,        0.0)  AS avg_adoption_score,
        COALESCE(ea.retention_signal_count,    0)    AS retention_signal_count,
        COALESCE(ea.integration_connects_first_30d, 0) AS integration_connects_first_30d,
        COALESCE(ta.high_priority_tickets,     0)    AS high_priority_tickets
    FROM customer_ref cr
    LEFT JOIN event_agg  ea ON cr.customer_id = ea.customer_id
    LEFT JOIN ticket_agg ta ON cr.customer_id = ta.customer_id
""").df()

print(f"Feature dataset: {len(feature_df):,} customers  |  "
      f"Churned: {feature_df['is_churned'].sum():,} ({feature_df['is_churned'].mean():.1%})")

In [ ]:
# ── Violin plots: churned vs. active ──────────────────────────────────────
feature_df["Status"] = feature_df["is_churned"].map({0: "Active", 1: "Churned"})
palette = {"Active": "#2196F3", "Churned": "#F44336"}

plot_features = [
    ("events_last_30d",             "Events (Last 30 Days)"),
    ("avg_adoption_score",          "Avg Feature Adoption Score"),
    ("retention_signal_count",      "Retention Signal Count"),
    ("high_priority_tickets",       "High-Priority Tickets (Lifetime)"),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 6))
for ax, (feat, title) in zip(axes, plot_features):
    sns.violinplot(
        data=feature_df, x="Status", y=feat, ax=ax,
        palette=palette, inner="quartile", cut=0,
    )
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("")

fig.suptitle(
    "Feature Distributions: Churned vs. Active Customers\n"
    "Visual proof that the data contains learnable signal for Phase 4 modelling",
    fontsize=13, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

print("✅ Key observations:")
for feat, title in plot_features:
    active_median = feature_df[feature_df["is_churned"]==0][feat].median()
    churned_median = feature_df[feature_df["is_churned"]==1][feat].median()
    direction = "↓ churned lower" if churned_median < active_median else "↑ churned higher"
    print(f"   {title:<35} active={active_median:.2f}, churned={churned_median:.2f}  {direction}")

## 1.5 — Feature Correlation Heatmap

> **Business Question:** Which features are most strongly associated with churn?
> Does the ranking align with our domain intuition?
>
> **Interpretation:** Point-biserial correlation is correct for a continuous feature
> vs. binary churn label. Negative r = higher feature → lower churn (protective).
> Positive r = higher feature → higher churn (risk signal).

This ranking feeds directly into Phase 4 feature engineering decisions.

In [ ]:

FEATURE_COLS = [
    "events_last_30d",
    "avg_adoption_score",
    "retention_signal_count",
    "integration_connects_first_30d",
    "high_priority_tickets",
]

# ── Point-biserial correlation vs. churn ──────────────────────────────────
corr_results = {}
for col in FEATURE_COLS:
    r, p = stats.pointbiserialr(feature_df[col], feature_df["is_churned"])
    corr_results[col] = {"correlation": round(r, 4), "p_value": round(p, 6), "|corr|": abs(r)}

corr_df = pd.DataFrame(corr_results).T.sort_values("|corr|", ascending=False)
print("=== Feature Correlations with Churn Label (ranked by |r|) ===")
print(corr_df[["correlation","p_value","|corr|"]].to_string())

top_3 = corr_df.head(3).index.tolist()
print(f"\nTop 3 features: {top_3}")

# Validate signal requirements
assert abs(corr_df.loc["events_last_30d", "correlation"]) > 0.30, "WARN: events_last_30d weak!"
assert "retention_signal_count" in top_3, "WARN: retention_signal_count not in top 3!"
print("\n✅ Signal validation passed")

In [ ]:
# ── Spearman correlation matrix ────────────────────────────────────────────
all_numeric = feature_df[FEATURE_COLS + ["is_churned"]].rename(
    columns={"is_churned": "CHURN_LABEL"}
)
spearman_matrix = all_numeric.corr(method="spearman")

fig, ax = plt.subplots(figsize=(8, 7))
mask = np.triu(np.ones_like(spearman_matrix, dtype=bool))
sns.heatmap(
    spearman_matrix, mask=mask,
    annot=True, fmt=".2f", cmap="coolwarm",
    center=0, vmin=-1, vmax=1, ax=ax,
    linewidths=0.5, annot_kws={"size": 10},
)
ax.set_title(
    "Spearman Correlation Matrix — Features × Churn Label",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
plt.show()

## 1.6 — The Integration Activation Gate

> **Business Question:** Is there a threshold number of integrations in the first 30 days
> that separates retained customers from at-risk ones?
>
> **Why this matters:** If integration_connect ≥ 3 is a reliable retention predictor,
> it becomes an actionable **onboarding milestone** — a binary gate CS can enforce.
> This converts an abstract engagement score into a concrete SOP.

In [ ]:
def integration_bucket(n: int) -> str:
    if n == 0:    return "0"
    elif n <= 2:  return "1–2"
    elif n <= 5:  return "3–5"
    else:         return "6+"

BUCKET_ORDER = ["0", "1–2", "3–5", "6+"]
feature_df["integration_bucket"] = feature_df["integration_connects_first_30d"].apply(integration_bucket)

bucket_stats = (
    feature_df.groupby("integration_bucket", observed=True)
    .agg(n_customers=("customer_id", "count"), n_churned=("is_churned", "sum"))
    .reset_index()
)
bucket_stats["churn_rate"] = bucket_stats["n_churned"] / bucket_stats["n_customers"]
bucket_stats["integration_bucket"] = pd.Categorical(
    bucket_stats["integration_bucket"], categories=BUCKET_ORDER, ordered=True
)
bucket_stats = bucket_stats.sort_values("integration_bucket")

print(bucket_stats.to_string(index=False))

# ── Bar chart ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
bar_colors = ["#D32F2F", "#F57C00", "#FBC02D", "#388E3C"]
bars = ax.bar(
    bucket_stats["integration_bucket"],
    bucket_stats["churn_rate"] * 100,
    color=bar_colors, edgecolor="white", linewidth=1.5, width=0.6,
)
for bar, (_, row) in zip(bars, bucket_stats.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.5,
        f"{row['churn_rate']:.1%}\n(n={row['n_customers']:,})",
        ha="center", va="bottom", fontsize=11, fontweight="bold",
    )

ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlabel("Integration Connects in First 30 Days", fontsize=12)
ax.set_ylabel("Churn Rate", fontsize=12)
ax.set_title(
    "Integration Activation Gate\nChurn Rate by Integration Count (First 30 Days)",
    fontsize=13, fontweight="bold",
)
ax.set_ylim(0, bucket_stats["churn_rate"].max() * 135)
plt.tight_layout()
plt.show()

# ── Business insight ───────────────────────────────────────────────────────
r0   = bucket_stats[bucket_stats["integration_bucket"] == "0"]["churn_rate"].values[0]
r3_5 = bucket_stats[bucket_stats["integration_bucket"] == "3–5"]["churn_rate"].values[0]
r6p  = bucket_stats[bucket_stats["integration_bucket"] == "6+"]["churn_rate"].values[0]
r3plus = (r3_5 + r6p) / 2

mrr_avg = customers_df["mrr"].mean()
n_zero = bucket_stats[bucket_stats["integration_bucket"] == "0"]["n_customers"].values[0]
arr_at_risk = n_zero * mrr_avg * 12 * r0

print(f"\n📊 Business Insight:")
print(f"   Churn rate (0 integrations):      {r0:.1%}")
print(f"   Churn rate (3+ integrations):     {r3plus:.1%}")
print(f"   Retention multiplier (0 vs 3+):   {r0/r3plus:.1f}×")
print(f"\n🎯 Exec Deck Bullet:")
print(f"   'Customers with ≥3 integrations in 30 days churn at {r3plus:.0%} vs {r0:.0%}.")
print(f"    That's a {r0/r3plus:.1f}× retention multiplier from a single onboarding milestone.'")
print(f"\n💰 ROI Connection:")
print(f"   {n_zero:,} customers have 0 integrations → ${arr_at_risk:,.0f} annual revenue at risk")

## Summary

| Finding | Evidence | Exec Deck Bullet |
|---|---|---|
| **Leaky bucket** | Month-3 dropout ~30% across cohorts | "~30% dropout by month 3 — CS window is now" |
| **Tier gap** | Starter ~43% lifetime churn, Enterprise ~7% | "6× churn gap — tiered CS investment justified" |
| **Adoption signal** | events_last_30d \|r\| > 0.30 | "Usage decay is the #1 leading indicator" |
| **Integration gate** | ≥3 integrations → 2.7× retention | "3 integrations in 30 days = the activation checkpoint" |

> **Next:** `phase3_02_survival_analysis.ipynb` — KM curves, Cox PH, and intervention window.

In [ ]:
conn.close()
print('Connection closed.')